# inference-mode-step — worked example 1: Decorate SGD step with inference_mode

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `inference-mode-step`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A hand-rolled optimizer mutates parameter leaves in place (`p -= lr * p.grad`). PyTorch forbids in-place ops on a leaf that requires grad, unless autograd is disabled. Decorating `step` with `@t.inference_mode()` disables graph tracking for the whole method, so the bare in-place update is legal and cheap.

## Worked solution

We build a minimal SGD whose `step` is allowed to mutate leaves.

1. `__init__` materializes `self.params = list(params)` (so a generator is not exhausted) and stores `self.lr`.
2. `step` is decorated with `@t.inference_mode()`. Inside, for each param with a gradient, we do `p -= self.lr * p.grad`. This is an in-place op on a leaf — only legal because the decorator turned off autograd tracking.
3. `zero_grad` sets every `p.grad = None`, the modern reset.
4. We fit a single weight toward a target for a few steps and confirm the loss decreases, demonstrating the update actually moved the parameter without raising the leaf-in-place error.

In [ ]:
import torch as t

t.manual_seed(0)

class InferenceSGD:
    def __init__(self, params, lr):
        self.params = list(params)
        self.lr = lr

    @t.inference_mode()
    def step(self):
        for p in self.params:
            if p.grad is not None:
                p -= self.lr * p.grad

    def zero_grad(self):
        for p in self.params:
            p.grad = None

w = t.tensor([5.0], requires_grad=True)
opt = InferenceSGD([w], lr=0.1)
losses = []
for _ in range(5):
    opt.zero_grad()
    loss = (w - 1.0) ** 2
    loss.backward()
    opt.step()
    losses.append(float(loss))
print('losses:', [round(l, 4) for l in losses])
print('decreasing:', losses[-1] < losses[0])